In [1]:
import numpy as np

X = np.array([[0,0],[0,1],[1,0],[1,1]])

y = np.array([0,0,0,1])

In [2]:
class Perceptron:
    def __init__(self, lr=0.1, epocas=10):
        self.lr = lr
        self.epocas = epocas

    def funcion_escalon(self, z):
        return np.where(z >= 0, 1, 0)

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0

        for ep in range(self.epocas):
            print(f"\nÉpoca {ep+1}")

            for i in range(len(X)):
                z = np.dot(X[i], self.w) + self.b
                y_pred = self.funcion_escalon(z)

                error = y[i] - y_pred

                self.w += self.lr * error * X[i]
                self.b += self.lr * error

                print(f"Pesos: {self.w}, Bias: {self.b}")

In [3]:
p = Perceptron(lr=0.1, epocas=10)
p.fit(X, y)


Época 1
Pesos: [0. 0.], Bias: -0.1
Pesos: [0. 0.], Bias: -0.1
Pesos: [0. 0.], Bias: -0.1
Pesos: [0.1 0.1], Bias: 0.0

Época 2
Pesos: [0.1 0.1], Bias: -0.1
Pesos: [0.1 0. ], Bias: -0.2
Pesos: [0.1 0. ], Bias: -0.2
Pesos: [0.2 0.1], Bias: -0.1

Época 3
Pesos: [0.2 0.1], Bias: -0.1
Pesos: [0.2 0. ], Bias: -0.2
Pesos: [0.1 0. ], Bias: -0.30000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004

Época 4
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004

Época 5
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004

Época 6
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004
Pesos: [0.2 0.1], Bias: -0.20000000000000004

Época 7
Pesos: [0.2 0.1], Bias: 

In [4]:
print("Frontera de decisión:")
print(f"{p.w[0]}*x1 + {p.w[1]}*x2 + {p.b} = 0")

Frontera de decisión:
0.2*x1 + 0.1*x2 + -0.20000000000000004 = 0


# El perceptrón converge correctamente para la compuerta AND
# porque es un problema linealmente separable

In [5]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_iris()
X = data.data
y = data.target

In [6]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [7]:
def one_hot(y, num_clases):
    return np.eye(num_clases)[y]

y_train_oh = one_hot(y_train, 3)

In [8]:
def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    return (x > 0).astype(float)

def softmax(x):
    exp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

In [9]:
class RedNeuronal:
    def __init__(self, capas, lr=0.01):
        self.lr = lr
        self.W = []
        self.b = []

        for i in range(len(capas)-1):
            self.W.append(np.random.randn(capas[i], capas[i+1]) * 0.1)
            self.b.append(np.zeros((1, capas[i+1])))

    def forward(self, X):
        self.A = [X]
        self.Z = []

        for i in range(len(self.W)-1):
            Z = np.dot(self.A[-1], self.W[i]) + self.b[i]
            A = relu(Z)
            self.Z.append(Z)
            self.A.append(A)

        Z = np.dot(self.A[-1], self.W[-1]) + self.b[-1]
        A = softmax(Z)

        self.Z.append(Z)
        self.A.append(A)

        return A

    def backward(self, y_true):
        m = y_true.shape[0]
        delta = self.A[-1] - y_true

        for i in reversed(range(len(self.W))):
            dW = np.dot(self.A[i].T, delta) / m
            db = np.sum(delta, axis=0, keepdims=True) / m

            if i > 0:
                delta = np.dot(delta, self.W[i].T) * relu_derivada(self.Z[i-1])

            self.W[i] -= self.lr * dW
            self.b[i] -= self.lr * db

    def entrenar(self, X, y, epocas=1000):
        for ep in range(epocas):
            self.forward(X)
            self.backward(y)

            if ep % 100 == 0:
                loss = -np.mean(np.sum(y * np.log(self.A[-1] + 1e-8), axis=1))
                print(f"Epoca {ep}, Loss: {loss}")

    def predecir(self, X):
        return np.argmax(self.forward(X), axis=1)

In [10]:
red = RedNeuronal([4, 8, 3], lr=0.01)
red.entrenar(X_train, y_train_oh, epocas=1000)

Epoca 0, Loss: 1.105620357092359
Epoca 100, Loss: 1.0623146600862658
Epoca 200, Loss: 0.979161916779914
Epoca 300, Loss: 0.8205649914932568
Epoca 400, Loss: 0.6557356279766169
Epoca 500, Loss: 0.5510135747290025
Epoca 600, Loss: 0.4871756293169041
Epoca 700, Loss: 0.4429100442004138
Epoca 800, Loss: 0.40877725047359176
Epoca 900, Loss: 0.38074300213240425


In [11]:
pred = red.predecir(X_test)
accuracy = np.mean(pred == y_test)

print("Accuracy:", accuracy)

Accuracy: 0.8333333333333334


# Se utilizó ReLU en capas ocultas y softmax en la salida.
# La función de pérdida es cross-entropy.
# La red logra buena precisión en el dataset Iris.